In [1]:

import math
import random
 
random.seed(42)

In [2]:
#CSV File Reader 
def read_csv(filepath):
    dataset = []
    with open(filepath, 'r') as f:
        lines = f.readlines()
    header = [col.strip() for col in lines[0].strip().split(",")]
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        values = line.split(",")
        row = {}
        for col,val in zip(header,values):
            row[col] = val.strip()
        dataset.append(row)
    print(f"[CSV] Loaded {len(dataset)} rows | {len(header)} columns")
    return dataset,header
    

In [3]:
# Exploratory Data Analysis
def describe_column(dataset,col,numeric=True):
    values = []
    for row in dataset:
        try:
            values.append(float(row[col]))
        except ValueError:
            pass
    if not values:
        return {}
    n = len(values)
    mean = sum(values)/n
    variance = sum((x - mean) ** 2 for x in values) / n
    std = math.sqrt(variance)
    mn = min(values)
    mx = max(values)
    sorted_v = sorted(values)
    median = sorted_v[n//2] if n%2 == 1 else (sorted_v[n//2-1] + sorted_v[n//2])/2
    return {"count":n,"mean": round(mean,4),"std": round(std,4),"min":mn,"median":median,"max":mx}

def value_counts(dataset,col):
    counts = {}
    for row in dataset:
        v = row[col]
        counts[v] = counts.get(v,0)+1
    return dict(sorted(counts.items(),key = lambda x: -x[1]))

def run_eda(dataset):
    print("\n"+"="*60)
    print("Exploratory Data Analysis")
    print("="*60)
    
    numeric_cols = ["age", "duration", "campaign", "pdays", "previous",
                    "emp_var_rate", "cons_price_idx", "cons_conf_idx",
                    "euribor3m", "nr_employed"]
    categorical_cols = ["job", "marital", "education", "default", "housing",
                        "loan", "contact", "month", "day_of_week", "poutcome"]
    print("\n---- Numeric Columns Statistics ----")
    for col in numeric_cols:
        stats = describe_column(dataset,col)
        print(f"  {col:20s} | mean={stats['mean']:>10.4f}  std={stats['std']:>10.4f}"
              f"  min={stats['min']:>8}  max={stats['max']:>8}")
    print("\n----Categorical Column value Counts----")
    for col in categorical_cols:
        vc = value_counts(dataset,col)
        top = list(vc.items())[:5]
        top_str = "  |  ".join(f"{k}:{v}" for k,v in top)
        print(f"  {col:15s} | {top_str}")
    
    vc_y = value_counts(dataset,"y")
    total = sum(vc_y.values())
    print("\n---Target Variable Distribution---")
    for k,v in vc_y.items():
        label = "Subscribed   (yes)" if k == "1" else "Not Subscribed(NO)"
        print(f"{label}:{v} ({100*v  /  total:..2f}%)")
        

In [ ]:
# Preprocessing 
CATEGOIRCAL_COLS = ["job", "marital", "education", "default", "housing",
                    "loan", "contact", "month", "day_of_week", "poutcome"]
NUMERIC_COLS = ["age", "duration", "campaign", "pdays", "previous",
                "emp_var_rate", "cons_price_idx", "cons_conf_idx", "euribor3m", "nr_employee"]
def build_label_encoders(dataset,cols):
    encoders = {}
    for col in cols:
        unique_vals = sorted(set(row[col] for row in dataset))
        encoders[col] = {val:idx for idx,val in enumerate(unique_vals)}
    return encoders

def compute_normalization_params(dataset,cols):
    params = {}
    for col in cols:
        values = [float(row[col]) for row in dataset]
        n = len(values)
        mean = sum(values)/n
        std = math.sqrt(sum((x - mean) ** 2 for x in values) / n)
        std = std if std != 0 else 1.0
        params[col] = (mean,std)
    return params

def preprocess(dataset,encoders,norm_params):
    X,y = [],[]
    for row in dataset:
        features = []
        for col in CATEGOIRCAL_COLS:
            features.append(float(encoders[col].get(row[col],0)))
        for col in NUMERIC_COLS:
            mean,std = norm_params[col]
            features.append((float(row[col])-mean)/std)
        X.append(features)
        y.append(float(row["y"]))
    return X,y

def train_test_split(X,y,test_ratio = 0.2):
    indices = list(range(len(X)))
    random.shuffle(indices)
    split = int(len(X)*(1-test_ratio))
    train_idx = indices[:split]
    test_idx = indices[split:]
    X_train = [X[i] for i in train_idx]
    y_train = [y[i] for i in train_idx]
    X_test = [X[i] for i in test_idx]
    y_test = [y[i] for i in test_idx]
    return X_train,y_train,X_test,y_test

    

In [5]:
#Metrics
def compute_metrices(y_true,y_pred):
    tp = sum(1 for a,b in zip(y_true,y_pred) if a == 1 and b == 1)
    tn = sum(1 for a,b in zip(y_true,y_pred) if a == 0 and b == 0)
    fp = sum(1 for a,b in zip(y_true,y_pred) if a == 0 and b == 1)
    fn = sum(1 for a,b in zip(y_true,y_pred) if a == 1 and b == 0)
    
    total = len(y_true)
    accuracy = (tp + tn) / total if total else 0 
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0 
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    
    return {"accuracy": round(accuracy,4),
            "precision": round(precision,4),
            "recall": round(recall,4),
            "f1": round(f1,4),
            "tp": tp,"tn": tn, "fp":fp, "fn":fn}
    
def print_metrics(name, metrics):
    print(f"\n  [{name}] Results")
    print(f"    Accuracy  : {metrics['accuracy']  * 100:.2f}%")
    print(f"    Precision : {metrics['precision'] * 100:.2f}%")
    print(f"    Recall    : {metrics['recall']    * 100:.2f}%")
    print(f"    F1 Score  : {metrics['f1']        * 100:.2f}%")
    print(f"    Confusion Matrix → TP:{metrics['tp']}  TN:{metrics['tn']}"
          f"  FP:{metrics['fp']}  FN:{metrics['fn']}")


In [6]:
# lOGISTIC REGRESSION MODEL
class LigusticRegression:
    def __init__(self,learning_rate=0.1,epochs=300):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = []
        self.bias = 0.0
    
    @staticmethod
    def sigmoid(z):
        z = max(-500,min(500,z))
        return 1.0 / (1.0 + math.exp(-z))
    def _dot(self,x):
        return sum(w * xi for w,xi in zip(self.weights,x)) + self.bias
    def fit(self,X,y):
        n_samples = len(X)
        n_features = len(X[0])
        self.weights = [0.0] * n_features
        self.bias = 0.0
        for epoch in range(self.epochs):
            predictions = [self.sigmoid(self._dot(x)) for x in X]
            errors = [p-yi for p, yi in zip(predictions,y)]
            dw = [0.0] * n_features
            db = 0.0
            
            for i in range(n_samples):
                for j in range(n_features):
                    dw[j] += errors[i] * X[i][j]
                db += errors[i]
            
            self.weights = [w - self.lr * (dw[j] / n_samples)
                            for j,w in enumerate(self.weights)]
            self.bias -= self.lr* (db/n_samples)
            
            if (epoch + 1) %50 == 0:
                loss = -sum(
                    yi * math.log(p + 1e-15) + (1-yi)*math.log(1-p+1e-9)
                    for p,yi in zip(predictions,y)
                ) / n_samples
                print(f"Epoch {epoch+1}/{self.epochs} - Loss: {loss:.4f}")
    
    def predict(self,X,threshold=0.5):
        return [1 if self._sigmoid(self._dot(x))>= threshold else 0 for x in X]
    def feature_importance(self,feature_names):
        pairs = sorted(zip(feature_names,self.weights),key = lambda x: abs(x[1]),reverse=True)
        return pairs
    
            

In [ ]:
# Decision Tree Classifier
